# Etapa 4 — A tabela de treino do antifraude

> **Trilha:** 1 · 2 · 3 · **4** · 5 · 6 · 7 — roteiro completo em `trilha/04_tabela_de_treino/README.md`

Agora você refaz, do jeito certo, a tabela que o notebook herdado usava.

| | |
|---|---|
| **Tabela** | `analytics.ds_pedido_fraude` |
| **Grão** | 1 linha por **pedido** |
| **Regra de ouro** | toda feature usa **só o que se sabia no momento da compra** |
| **Rótulo** | `is_fraude`, só quando já está **maduro** |
| **Divisão** | coluna `particao`, **temporal**: treino no passado, teste depois |

No fim, a prova de fogo: o mesmo `avaliar` que derrubou o notebook herdado.

In [ ]:
from liga import sql, executar, conferir, avaliar, progresso

## 4.1 O grão: agregar os itens antes de juntar

In [ ]:
sql("""
SELECT count(*)                  AS linhas_na_fato_pedidos,
       count(DISTINCT id_pedido) AS pedidos
FROM fato_pedidos
""")

A solução é **agregar primeiro**: `GROUP BY id_pedido` na `fato_pedidos`, somando o valor
dos itens, contando os itens e pegando o aparelho, a UF e o pagamento (iguais em todos os
itens do mesmo pedido). Só **depois** se junta com as dimensões. Assim cada junção
encontra uma linha por pedido, e nada se multiplica.

## 4.2 O rótulo e o tempo

Três datas organizam a tabela inteira:

In [ ]:
sql("""
SELECT max(dt_venda)::date                        AS foto,
       max(dt_venda)::date - 60                   AS fim_do_treino,
       max(dt_venda)::date - 7                    AS fim_do_teste
FROM fato_pedidos
""")

| Pedidos feitos... | `particao` | `is_fraude` | Por quê |
|---|---|---|---|
| antes de `fim_do_treino` | `treino` | o rótulo | o passado: onde o modelo aprende |
| entre os dois | `teste` | o rótulo | o "futuro" do modelo: onde ele é avaliado |
| nos últimos 7 dias | `sem_rotulo` | **nulo** | a fraude leva até 4 dias para ser descoberta: o rótulo ainda não amadureceu |

A divisão é **temporal**, nunca sorteada: em produção o modelo sempre prevê o futuro com
o passado. E o rótulo imaturo vira **nulo**, não "falso": "não sabemos ainda" é diferente
de "não é fraude".

## 4.3 O aparelho, sem futuro

O notebook herdado usava `qtd_contas` da `dim_dispositivo` (o total até a foto). Veja a
diferença, para os pedidos feitos com aparelhos de anel de fraude:

In [ ]:
sql("""
WITH uso AS (
    SELECT p.id AS id_pessoa, f.sk_dispositivo, min(f.dt_venda) AS primeiro_uso
    FROM fato_pedidos f JOIN dim_pessoas p ON p.sk_pessoa = f.sk_pessoa
    GROUP BY 1, 2
),
pedidos AS (
    SELECT DISTINCT f.id_pedido, f.dt_venda, f.sk_dispositivo, p.id AS id_pessoa
    FROM fato_pedidos f JOIN dim_pessoas p ON p.sk_pessoa = f.sk_pessoa
)
SELECT d.device_id, pd.id_pedido, pd.dt_venda,
       d.qtd_contas AS notebook_herdado_qtd_contas,
       (SELECT count(*) FROM uso u
         WHERE u.sk_dispositivo = pd.sk_dispositivo AND u.id_pessoa <> pd.id_pessoa
           AND u.primeiro_uso < pd.dt_venda) AS outras_contas_ate_a_compra
FROM pedidos pd
JOIN dim_dispositivo d ON d.sk_dispositivo = pd.sk_dispositivo
WHERE d.device_id = 'd-2ad820ca'
ORDER BY pd.dt_venda
LIMIT 8
""")

Nas primeiras compras do anel, o aparelho ainda era de uma conta só: `0` outras contas.
O número cresce compra a compra. O notebook herdado dava ao modelo o **5 final** desde a
primeira compra.

A mesma ideia vale para `is_dispositivo_novo` ("a pessoa nunca tinha usado este aparelho
**antes desta compra**").

## 4.4 O cliente, no dia da compra

As features do cliente vêm da `fs_cliente` da etapa 3, juntada **pelo dia da compra**:

```sql
JOIN analytics.fs_cliente fs ON fs.id_pessoa = pe.id AND fs.data = pd.dt_venda::date
```

É o *as-of join* de novo, agora usando a *feature table*: "como era este cliente no
início do dia em que comprou". Repare que `taxa_fraude_cliente` **não existe** aqui: a
única forma honesta dela exigiria saber quais pedidos anteriores **já tinham sido
confirmados** como fraude no momento da compra — um bom desafio para depois.

## 4.5 Construindo a tabela

In [ ]:
executar("""
DROP TABLE IF EXISTS analytics.ds_pedido_fraude;

CREATE TABLE analytics.ds_pedido_fraude AS
WITH foto AS (
    -- o dia em que o DW foi fotografado: a referencia de "hoje"
    SELECT max(dt_venda)::date AS dia FROM public.fato_pedidos
),
pedidos AS (
    -- os ITENS viram PEDIDO: agregar antes de juntar, para nao multiplicar linhas
    SELECT f.id_pedido,
           min(f.sk_pessoa)          AS sk_pessoa,
           min(f.dt_venda)           AS dt_venda,
           sum(f.valor_total)        AS valor_total,
           count(*)                  AS qtd_itens,
           sum(f.peso_gramas)        AS peso_gramas,
           min(f.sk_forma_pagamento) AS sk_forma_pagamento,
           min(f.sk_localidade)      AS sk_localidade,
           min(f.sk_dispositivo)     AS sk_dispositivo,
           bool_or(f.is_fraude)      AS is_fraude,
           max(f.dt_fraude)          AS dt_fraude
    FROM public.fato_pedidos f
    GROUP BY f.id_pedido
),
uso_aparelho AS (
    -- quando cada pessoa usou cada aparelho pela primeira vez
    SELECT p.id AS id_pessoa, f.sk_dispositivo, min(f.dt_venda) AS primeiro_uso
    FROM public.fato_pedidos f
    JOIN public.dim_pessoas p ON p.sk_pessoa = f.sk_pessoa
    GROUP BY 1, 2
)
SELECT pd.id_pedido,
       pe.id                                             AS id_pessoa,
       pd.dt_venda,
       pd.valor_total,
       pd.qtd_itens::integer                             AS qtd_itens,
       round(pd.peso_gramas / 1000.0, 2)                 AS peso_kg,
       fp.forma                                          AS forma_pagamento,
       extract(hour FROM pd.dt_venda)::integer           AS hora_venda,
       t.is_fim_semana,
       NOT l.is_area_atendida                            AS is_fora_da_area,
       l.uf <> fs.uf                                     AS is_uf_diferente_do_cadastro,
       -- o aparelho: so o que se sabia ANTES desta compra
       NOT EXISTS (SELECT 1 FROM uso_aparelho u
                   WHERE u.id_pessoa = pe.id AND u.sk_dispositivo = pd.sk_dispositivo
                     AND u.primeiro_uso < pd.dt_venda)   AS is_dispositivo_novo,
       (SELECT count(*) FROM uso_aparelho u
         WHERE u.sk_dispositivo = pd.sk_dispositivo AND u.id_pessoa <> pe.id
           AND u.primeiro_uso < pd.dt_venda)::integer    AS qtd_outras_contas_no_dispositivo,
       -- o cliente: como ele era no inicio do dia da compra (feature table)
       fs.dias_de_cadastro,
       fs.qtd_pedidos_total,
       fs.qtd_pedidos_30d,
       fs.valor_medio_pedido,
       fs.dias_desde_ultimo_pedido,
       fs.qtd_dispositivos,
       round(pd.valor_total / NULLIF(fs.valor_medio_pedido, 0), 2) AS razao_valor_vs_media,
       -- o rotulo, so quando ja esta maduro
       CASE WHEN pd.dt_venda < foto.dia - 7 THEN pd.is_fraude END  AS is_fraude,
       CASE WHEN pd.dt_venda < foto.dia - 60 THEN 'treino'
            WHEN pd.dt_venda < foto.dia - 7  THEN 'teste'
            ELSE 'sem_rotulo' END                                  AS particao
FROM pedidos pd
CROSS JOIN foto
JOIN public.dim_pessoas pe          ON pe.sk_pessoa = pd.sk_pessoa
JOIN public.dim_forma_pagamento fp  ON fp.sk_forma_pagamento = pd.sk_forma_pagamento
JOIN public.dim_localidade l        ON l.sk_localidade = pd.sk_localidade
JOIN public.dim_tempo t             ON t.data = pd.dt_venda::date
JOIN analytics.fs_cliente fs        ON fs.id_pessoa = pe.id AND fs.data = pd.dt_venda::date;

ALTER TABLE analytics.ds_pedido_fraude ADD PRIMARY KEY (id_pedido);
""")

## 4.6 A prova

In [ ]:
conferir("ds_pedido_fraude")

## 4.7 A prova de fogo

O mesmo teste que derrubou o notebook herdado, agora com a sua tabela:

In [ ]:
avaliar("fraude")

Compare com o notebook herdado:

In [ ]:
avaliar("legado")

O seu modelo parece **pior no notebook** do que o herdado — e é **muito melhor em
produção**. E,
mais importante: o número que você vê no notebook é o número que a empresa vai ter de
verdade. É isso que uma boa modelagem de dados entrega a um cientista de dados: **um
número em que se pode confiar**.

## Os sete erros, corrigidos

| # | Erro do notebook herdado | Como a sua tabela resolve |
|---|---|---|
| 1 | linhas por item e por versão | agrega os itens antes de juntar; junta a pessoa pela `sk` |
| 2 | o mesmo pedido no treino e no teste | `particao` temporal, uma linha por pedido |
| 3 | UF de hoje | UF de `fs_cliente` no dia da compra |
| 4 | `taxa_fraude_cliente` com o rótulo | não existe; o histórico vem da `fs_cliente`, só com o passado |
| 5 | `status` de depois | não existe: o modelo só vê o que existe na compra |
| 6 | `qtd_contas` final | `qtd_outras_contas_no_dispositivo` até a compra |
| 7 | rótulo imaturo | `is_fraude` nulo nos últimos 7 dias |

In [ ]:
progresso()

✅ **Etapa 4 concluída.** Esta é a etapa central da disciplina.

➡️ Continue em **`05_painel.ipynb`**: outro tipo de tabela, outra armadilha.